In [1]:
!nvidia-smi

Sun Aug 23 23:43:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install nvcc4jupyter

In [4]:
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmprsp0fh79".


In [19]:
%%cuda
#include <iostream>

// This is the "Kernel" - it runs directly on the T4 GPU cores
__global__ void helloFromGPU() {
    auto threadId = threadIdx.x;
    auto blockId = blockIdx.x;
    printf("Hello from T4 GPU! Block: %d, Thread: %d\n", blockId, threadId);
}

int main() {
    std::cout << "Hello from the CPU!" << std::endl;

    // Launch the function on the GPU: 2 blocks, 4 threads per block (8 parallel threads total)
    helloFromGPU<<<2, 4>>>();

    // Tell the CPU to wait for the GPU to finish printing before exiting
    cudaDeviceSynchronize();

    return 0;
}

Hello from the CPU!
Hello from T4 GPU! Block: 0, Thread: 0
Hello from T4 GPU! Block: 0, Thread: 1
Hello from T4 GPU! Block: 0, Thread: 2
Hello from T4 GPU! Block: 0, Thread: 3
Hello from T4 GPU! Block: 1, Thread: 0
Hello from T4 GPU! Block: 1, Thread: 1
Hello from T4 GPU! Block: 1, Thread: 2
Hello from T4 GPU! Block: 1, Thread: 3



In [90]:
%%cuda
#include <iostream>
#include <chrono>
#include <cstdint>
#include <algorithm>

void cpuVecAdd(float* A_h, float* B_h, float* C_h, uint64_t n)
{
    for (uint64_t i = 0; i < n; i++)
    {
        C_h[i] = A_h[i] + B_h[i];
    }
}

__global__ void vecAddKernel(float* A, float* B, float* C, uint64_t n)
{
    int i = threadIdx.x + blockIdx.x * blockDim.x;

    if (i < n)
        C[i] = A[i] + B[i];
}

void cudaVecAdd(float* A_h, float* B_h, float* C_h, uint64_t n)
{
    uint64_t size = n * sizeof(float);
    float *A_d, *B_d, *C_d;

    cudaMalloc((void**) &A_d, size);
    cudaMalloc((void**) &B_d, size);
    cudaMalloc((void**) &C_d, size);

    cudaMemcpy(A_d, A_h, size, cudaMemcpyHostToDevice);
    cudaMemcpy(B_d, B_h, size, cudaMemcpyHostToDevice);

    vecAddKernel<<<std::ceil(n/256.0), 256>>>(A_d, B_d, C_d, n);
    
    cudaMemcpy(C_h, C_d, size, cudaMemcpyDeviceToHost);
    
    cudaFree(A_d);    
    cudaFree(B_d);    
    cudaFree(C_d);    
}

int main()
{
    float *A, *B, *C;   
    uint64_t max = 2560; // t4 gpu core size

    for (uint64_t N = 10; N < max; N += 10)
    {
        uint64_t size = N * sizeof(float);
        std::cout << "N size: " << N << std::endl;

        A = (float*)malloc(size);
        if(!A)
        {    
            std::cout << "ERROR IN A: Malloc failed" << std::endl;
            return -1;
        }

        B = (float*)malloc(size);
        if(!B)
        {    
            std::cout << "ERROR IN B: Malloc failed" << std::endl;
            return -1;
        }

        C = (float*)malloc(size);
        if(!C)
        {    
            std::cout << "ERROR IN C: Malloc failed" << std::endl;
            return -1;
        }    

        std::cout << "Before For Loop: " << std::endl;

        for (int i = 0; i < N; i++)
        {
            A[i] = i;
            B[i] = 2 * i;
        }

        auto t0 = std::chrono::steady_clock::now();
        cpuVecAdd(A,B,C,N);
        auto t1 = std::chrono::steady_clock::now();
        std::chrono::duration<double, std::milli> ms = t1 - t0;
        std::cout << "Cpu-VecAdd: " << ms.count() << " ms" << std::endl;

        auto t3 = std::chrono::steady_clock::now();
        cudaVecAdd(A,B,C,N);
        auto t4 = std::chrono::steady_clock::now();
        std::chrono::duration<double, std::milli> cudaMs = t4 - t3;
        std::cout << "Cuda-VecAdd: " << cudaMs.count() << " ms" << std::endl;

        std::cout << N << " Success\n" << std::endl;

        if(A)
            free(A);
        if(B)
            free(B);
        if(C)
            free(C);
    }
}

N size: 10
Before For Loop: 
Cpu-VecAdd: 0.000163 ms
Cuda-VecAdd: 282.538 ms
10 Success

N size: 20
Before For Loop: 
Cpu-VecAdd: 0.000173 ms
Cuda-VecAdd: 0.218106 ms
20 Success

N size: 30
Before For Loop: 
Cpu-VecAdd: 0.000199 ms
Cuda-VecAdd: 0.217003 ms
30 Success

N size: 40
Before For Loop: 
Cpu-VecAdd: 0.000202 ms
Cuda-VecAdd: 0.196472 ms
40 Success

N size: 50
Before For Loop: 
Cpu-VecAdd: 0.000229 ms
Cuda-VecAdd: 0.191645 ms
50 Success

N size: 60
Before For Loop: 
Cpu-VecAdd: 0.000322 ms
Cuda-VecAdd: 0.184891 ms
60 Success

N size: 70
Before For Loop: 
Cpu-VecAdd: 0.000323 ms
Cuda-VecAdd: 0.201451 ms
70 Success

N size: 80
Before For Loop: 
Cpu-VecAdd: 0.000367 ms
Cuda-VecAdd: 0.205957 ms
80 Success

N size: 90
Before For Loop: 
Cpu-VecAdd: 0.000376 ms
Cuda-VecAdd: 0.182217 ms
90 Success

N size: 100
Before For Loop: 
Cpu-VecAdd: 0.000432 ms
Cuda-VecAdd: 0.181201 ms
100 Success

N size: 110
Before For Loop: 
Cpu-VecAdd: 0.000473 ms
Cuda-VecAdd: 0.191291 ms
110 Success

N size: